In [ ]:
18*18*6

In [ ]:
import numpy as np
import xarray as xr
import cftime
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.patches import Rectangle
import matplotlib.colors as mcolors


import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

In [ ]:
mlhb = xr.open_dataset('./data-20c/dTdt.nc').MLHB.assign_coords(time=pd.date_range('1900-01-01', '2010-12-01', freq='MS'))
cp = np.fromfile('./data/CP.txt', dtype='<U48').astype('datetime64[M]')
ep = np.fromfile('./data/EP.txt', dtype='<U48').astype('datetime64[M]')
mix = np.fromfile('./data/MIX.txt', dtype='<U48').astype('datetime64[M]')
suc = np.fromfile('./data/SUC.txt', dtype='<U48').astype('datetime64[M]')
cp = cp[cp < np.datetime64('2010-01')]
ep = ep[ep < np.datetime64('2010-01')]
mix = mix[mix < np.datetime64('2010-01')]
suc = suc[suc < np.datetime64('2010-01')]
events = [mix, ep, cp, suc]

In [ ]:
lat = mlhb.lat.values
lon = mlhb.lon.values
month = np.arange(-2, 22)
nt, ny, nx = len(month), len(lat), len(lon)

ds_nino = xr.Dataset(
    {
        'mlhb':(['type', 'month', 'term', 'lat', 'lon'], np.empty((4, nt, 15, ny, nx))),
    },
    coords={
        'type':(['type'], ['MIX', 'EP', 'CP', 'SUC']),
        'month':(['month'], month),
        'term':(['term'], mlhb.term.data),
        'lat':(['lat'], lat),
        'lon':(['lon'], lon)
    }
)
for i in range(4):
    for j in range(nt):
        ds_nino.mlhb[i, j] = mlhb.loc[(events[i] + np.timedelta64(month[j], "M")).astype('datetime64[ns]')].mean(dim='time').values





In [ ]:
mlhb.term.data

In [ ]:
ds_nino.mlhb[2, :12, 10:14].sum(dim='term').plot.contourf(col='month', col_wrap=3, levels=np.linspace(-2, 2, 21), aspect=2, size=2)

In [ ]:
ds_nino.mlhb[2, :12, 1:4].sum(dim='term').plot.contourf(col='month', col_wrap=3, levels=np.linspace(-2, 2, 21), aspect=2, size=2)

In [ ]:
ds_nino.mlhb[2, :12, 1:-1].sum(dim='term').plot.contourf(col='month', col_wrap=3, levels=np.linspace(-2, 2, 21), aspect=2, size=2)

In [ ]:
ds_nino.mlhb[2, :12, -1].plot.contourf(col='month', col_wrap=3, levels=np.linspace(-2, 2, 21), aspect=2, size=2)

In [ ]:
ds_nino.mlhb[:, :, 0, :, :].loc[:, :, -5:5, :].mean(dim='lat').plot.contourf(col='type', col_wrap=2, levels=21)

In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(12, 12))
for i in range(4):
    ax = axs[i]
    x = np.arange(15)
    y1 = ds_nino.mlhb[i, :8, :,].mean(dim='month').loc[:, -5:5, 160:210].mean(dim=['lat', 'lon'])
    y1[-1] = y1[0] - y1[1:-1].sum()
    y2 = ds_nino.mlhb[i, :8, :,].mean(dim='month').loc[:, -5:5, 210:270].mean(dim=['lat', 'lon'])
    y2[-1] = y2[0] - y2[1:-1].sum()
    y = y1 + y2
    width = 0.2
    ax.bar(x-0.2, y, width=width, color='orange', edgecolor='white', linewidth=0.5, label='160-270')
    ax.bar(x, y1, width=width, color='orchid', edgecolor='white', linewidth=0.5, label='160-210')
    ax.bar(x+0.2, y2, width=width, color='pink', edgecolor='white', linewidth=0.5, label='210-270')
    ax.legend()
    ax.set_xticks(x)
    ax.set_xticklabels(ds_nino.term.data, fontsize=8)
    ax.axhline(y=0, color='black', linewidth=0.8, linestyle='-', zorder=5)
    ax.set_ylim([-0.4, 0.6])
    ax.set_title(ds_nino.type.data[i], fontsize=10)
    
fig.subplots_adjust(left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.3, hspace=0.3)
fig.savefig('./pic/bar.png', dpi=300)

In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(12, 12))
for i in range(4):
    ax = axs[i]
    x = np.arange(15)
    y1 = ds_nino.mlhb[i, 18:28, :,].mean(dim='month').loc[:, -5:5, 170:220].mean(dim=['lat', 'lon'])
    # y1[-1] = y1[0] - y1[1:-1].sum()
    y2 = ds_nino.mlhb[i, 18:28, :,].mean(dim='month').loc[:, -5:5, 220:270].mean(dim=['lat', 'lon'])
    # y2[-1] = y2[0] - y2[1:-1].sum()
    y = y1 + y2
    width = 0.2
    ax.bar(x-0.2, y, width=width, color='orange', edgecolor='white', linewidth=0.5, label='160-270')
    ax.bar(x, y1, width=width, color='orchid', edgecolor='white', linewidth=0.5, label='160-210')
    ax.bar(x+0.2, y2, width=width, color='pink', edgecolor='white', linewidth=0.5, label='210-270')
    ax.legend()
    ax.set_xticks(x)
    ax.set_xticklabels(ds_nino.term.data[:], fontsize=8)
    ax.axhline(y=0, color='black', linewidth=0.8, linestyle='-', zorder=5)
    ax.set_ylim([-0.4, 0.6])
    ax.set_title(ds_nino.type.data[i], fontsize=10)
    
fig.subplots_adjust(left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.3, hspace=0.3)
# fig.savefig('./pic-20c/bar.png', dpi=300)

In [ ]:
def plot_sst(axs, data, levels):
    for i, ax in enumerate(axs.flatten()):
        ax.set_title(titles[i])
        cb = ax.contourf(lats, mon, data[i], levels=levels, cmap='bwr')
        # ax.set_yticks(np.arange(1, 25, 4))
        # ax.set_yticklabels(months_str[np.arange(1, 25, 4)], fontsize=9) 
        # ax.set_xticks(np.arange(120, 281, 40))
    return cb

fig, axs = plt.subplots(4, 4, figsize=(12, 7))

titles = ['MIX', 'EP', 'CP', 'SUC']
months_str = np.array(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
months_str = np.array([f'{months_str[i]} 0' for i in range(12)] + [f'{months_str[i]} 1' for i in range(12)]) 
mon = month
lon = ds_nino.mlhb.lon.values

levels = np.linspace(-1.5, 1.5, 21)
start, end = -5, 5
# cb = plot_sst(axs[0, :], ds_nino.mlhb[:, :, 1:3].sum(dim='term').loc[:, :, start:end, :].mean(dim='lat'), levels=levels)
# cb = plot_sst(axs[1, :], ds_nino.mlhb[:, :, 4:5].sum(dim='term').loc[:, :, start:end, :].mean(dim='lat'), levels=levels)
# cb = plot_sst(axs[2, :], ds_nino.mlhb[:, :, 7:10].sum(dim='term').loc[:, :, start:end, :].mean(dim='lat'), levels=levels)
# cb = plot_sst(axs[3, :], ds_nino.mlhb[:, :, 10:14].sum(dim='term').loc[:, :, start:end, :].mean(dim='lat'), levels=levels)

lons = np.arange(200, 230)
lats = 20/30*(lons-200)
cb = plot_sst(axs[0, :], 
              ds_nino.mlhb[:, :, 2].interp(lat=lats, lon=lons).sel(lat=xr.DataArray(lats, coords={'points':lats}), lon=xr.DataArray(lons, coords={'points':lats})),
             levels=levels)
cb = plot_sst(axs[1, :], 
              ds_nino.mlhb[:, :, 4].interp(lat=lats, lon=lons).sel(lat=xr.DataArray(lats, coords={'points':lats}), lon=xr.DataArray(lons, coords={'points':lats})),
             levels=levels)
cb = plot_sst(axs[2, :], 
              ds_nino.mlhb[:, :, 7:10].sum(dim='term').interp(lat=lats, lon=lons).sel(lat=xr.DataArray(lats, coords={'points':lats}), lon=xr.DataArray(lons, coords={'points':lats})),
             levels=levels)
cb = plot_sst(axs[3, :], 
              ds_nino.mlhb[:, :, 10:14].sum(dim='term').interp(lat=lats, lon=lons).sel(lat=xr.DataArray(lats, coords={'points':lats}), lon=xr.DataArray(lons, coords={'points':lats})),
             levels=levels)
cbaxes = fig.add_axes([0.9, 0.2, 0.01, 0.6])
cb = plt.colorbar(cb, cax=cbaxes, orientation="vertical")

fig.subplots_adjust(left=0.07, right=0.85, bottom=0.05, top=0.95, wspace=0.4, hspace=0.4) 
# fig.savefig('./pic-20c/dTdt-time-lon.png')

In [ ]:
ds = xr.open_dataset('./data-20c/composite-nino.nc')
sst = ds.t
sst_idx = sst.loc[:, :, -5:5, 180:220].mean(dim=['lat', 'lon'])

In [ ]:
xr.corr(ds_nino.mlhb[2, :, 4:7].sum(dim='term'), sst_idx[2, :], dim='month').plot()

In [ ]:
xr.corr(ds_nino.mlhb[2, 2:, 10:14].sum(dim='term'), sst_idx[2, :-2], dim='month').plot()

In [ ]:
sst_idx[2]